# Example: Sensitivity of Coupon Treasury Notes and Bonds
In this example, we use a controlled coupon-bond pricing experiment to study the asymmetric and coupon-dependent price responses described by Malkiel's fourth and fifth bond-pricing theorems.

> __Learning Objectives:__
>
> By the end of this example, you will be able to:
>
> * __Price coupon Treasury securities:__ Compute note and bond prices from their coupon and principal cash flows under a stated yield convention.
> * __Demonstrate convex price response:__ Show that an equal decrease in yield raises price more than an equal increase in yield lowers it.
> * __Evaluate coupon sensitivity:__ Compare percentage price changes across coupon rates while holding the other model inputs fixed.
> * __Interpret the evidence correctly:__ Distinguish a numerical implication of the assumed cash-flow model from an empirical validation using independent market data.

Let's use the model to examine Malkiel's fourth and fifth theorems.
___


## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's set up our code environment:

In [1]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

  Activating project at `~/Desktop/julia_work/CHEME-5660-CourseRepository-Fall-2026`


For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/) and the [CHEME 5660 Quantitative Finance Package documentation](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/).

Before we do any pricing calculations let's construct an instance of the [`DiscreteCompoundingModel` type](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/fixed/#VLQuantitativeFinancePackage.DiscreteCompoundingModel) and store this discount model in the `discount_model` variable.

> The [`DiscreteCompoundingModel` type](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/fixed/#VLQuantitativeFinancePackage.DiscreteCompoundingModel) has no data associated with it. Instead, it is used [by the Julia multiple dispatch system](https://docs.julialang.org/en/v1/manual/methods/#Methods) so that we call the correct pricing methods later.

Let's build our discount model:

In [2]:
discount_model = DiscreteCompoundingModel();

___

## Task 1: Compute the price and cash flow of a note
In this task, we'll demonstrate how to compute the price of a coupon-bearing Treasury note.

First, let's build an [instance of the `MyUSTreasuryCouponSecurityModel` type](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/fixed/#VLQuantitativeFinancePackage.MyUSTreasuryCouponSecurityModel) using [a custom `build(...)` method](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/fixed/#VLQuantitativeFinancePackage.build-Tuple{Type{MyUSTreasuryCouponSecurityModel},%20NamedTuple}), then we'll compute the auction price.

> __Example.__ Compute the price and cashflow for a T = 7-yr note, with a coupon rate of `c = 1.375%`, a yield (discount rate) rate = 1.461%, two coupon payments per year ($\lambda = 2$) and a face (par) value of $V_{P}$ = 100 USD. The price value reported on [TreasuryDirect.gov](https://www.treasurydirect.gov/marketable-securities/understanding-pricing/#id-for-more-detailed-formulas-and-useful-tables-264977) for this note is $V_{B}$ = `99.4299 USD`.

Similar to zero-coupon T-bills, we'll use a `short-cut` syntax that relies on the [Julia piping |> operator](https://docs.julialang.org/en/v1/manual/functions/#Function-composition-and-piping) and some syntax sugar to compute the coupon-bearing price, and discount factors and the cashflow for the instrument.

Let's store the result in the `test_note::MyUSTreasuryCouponSecurityModel` variable.

In [3]:
test_note = let

    ### BEGIN SOLUTION
    T = 7 # maturity in years
    y = 0.01461 # yield 
    c = 0.01375 # coupon rate
    ### END SOLUTION

    test_note = build(MyUSTreasuryCouponSecurityModel, (
        T = T, rate = y, coupon = c, λ = 2, par = 100.0
    )) |> discount_model;

    test_note; # return
end;

We have populated [the data fields of the `MyUSTreasuryCouponSecurityModel` instance](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/fixed/#VLQuantitativeFinancePackage.MyUSTreasuryCouponSecurityModel). So now let's pull the price, discount, and cash flow fields from `test_note::MyUSTreasuryCouponSecurityModel` and store them in the `nominal_computed_price,` `cashflow,` and `discount` variables:

In [4]:
nominal_computed_price = test_note.price;
cashflow = test_note.cashflow;
discount = test_note.discount;
println("This nominal computed note price = $(nominal_computed_price) USD")

This nominal computed note price = 99.42973596186266 USD


Did we get the same price as that recorded at auction?

> __Test:__ If two values are within some relative tolerance `rtol` of each other, the [isapprox function](https://docs.julialang.org/en/v1/base/math/#Base.isapprox) returns `true`; otherwise, it returns `false`. When the [isapprox function](https://docs.julialang.org/en/v1/base/math/#Base.isapprox) is combined with the [@assert macro](https://docs.julialang.org/en/v1/base/base/#Base.@assert), a `false` result generates an [AssertionError](https://docs.julialang.org/en/v1/base/base/#Core.AssertionError). Thus, we have a simple test for equality that works well for floating-point numbers.

So, how did we do?

In [5]:
observed_bond_price = 99.4299;
@assert isapprox(observed_bond_price, nominal_computed_price; rtol = 1e-4);

### Visualize the cash flows
`Unhide` the code block below to see how we build a table holding the nominal, discounted, and cumulative cash flow for this note using the [PrettyTables.jl package](https://github.com/ronisbr/PrettyTables.jl). We'll iterate through each period [using a `for-loop`](https://docs.julialang.org/en/v1/base/base/#for) and populate the `note_data_table::Array{Any,2}` variable.

For each iteration of the loop:
> __At each iteration:__ We access values for the discount and cashflow in period `i` and compute the cumulative cashflow in the `cumulative_payment` variable. We then store these data along with the nominal cash flow for each period in the `note_data_table` array

We display the data in the `note_data_table` variable by calling [the `pretty_table(...)` function exported by the PrettyTables.jl package](https://github.com/ronisbr/PrettyTables.jl) (with optional values for the `header,` and `tf` arguments).

See the [PrettyTables.jl package documentation for information about the `pretty_table(...)` function](https://ronisbr.github.io/PrettyTables.jl/stable/).

In [6]:
    let

        # initalize -
        number_of_periods = length(cashflow)
        note_data_table = Array{Any,2}(undef, number_of_periods, 5);
        cumulative_payment = 0.0;
        for i ∈ 0:(number_of_periods - 1)
            
            discount_value = discount[i]
            payment = cashflow[i];
            cumulative_payment += payment;

            note_data_table[i+1,1] = i;
            note_data_table[i+1,2] = discount_value;
            note_data_table[i+1,3] = discount_value*payment;
            note_data_table[i+1,4] = payment;
            note_data_table[i+1,5] = cumulative_payment;
        end

        # show the table -
        pretty_table(note_data_table,
            column_labels=["Period", "Discount factor", "Nominal cash flow", "Discounted cash flow", "Cumulative cash flow"], 
            table_format = TextTableFormat(borders = text_table_borders__simple))

    end

========= ================= =================== ====================== =========
  Period   Discount factor   Nominal cash flow   Discounted cash flow   Cumula ⋯
========= ================= =================== ====================== =========
       0               1.0            -99.4297               -99.4297          ⋯
       1            1.0073              0.6875               0.682514          ⋯
       2           1.01466              0.6875               0.677565          ⋯
       3           1.02208              0.6875               0.672651          ⋯
       4           1.02954              0.6875               0.667773          ⋯
       5           1.03706              0.6875                0.66293          ⋯
       6           1.04464              0.6875               0.658123          ⋯
       7           1.05227              0.6875                0.65335          ⋯
       8           1.05996              0.6875               0.648612          ⋯
       9            1.0677  

For this experiment, let's use the `test_note::MyUSTreasuryCouponSecurityModel` instance from Task 1, however, change the maturity from 7 years to 20 years. The price of the hypothetical 20-year bond is $V_{B}=$ `98.51 USD`.

First, specify the number (odd) of perturbation values in the `number_of_samples_theorem_4` variable. Next, specify the lower bound in the `β₁` variable and the upper bound in the `β₂` variable. Finally, compute the perturbation array (stored in the `β_theorem_4::Array{Float64,1}` variable) using the [range function](https://docs.julialang.org/en/v1/base/math/#Base.range) in combination with the [Julia pipe |> operator](https://docs.julialang.org/en/v1/manual/functions/#Function-composition-and-piping), and the [collect function](https://docs.julialang.org/en/v1/base/collections/#Base.collect-Tuple{Type,%20Any}).

In [7]:
number_of_samples_theorem_4  = 7;
β₁ = 0.8;
β₂ = 1.2;
β_theorem_4 = range(β₁, stop = β₂, length = number_of_samples_theorem_4) |> collect;

Your job is to complete the implementation of the `Theorem 4` simulation started below and analyze the simulation results. Let's display the results in a table using the `pretty_table(...)` function exported from the [PrettyTables.jl package](https://github.com/ronisbr/PrettyTables.jl)

In [8]:
let
    VB20 = 98.51317476187917;
    simulation_results_thm4_array = Array{Float64,2}(undef, number_of_samples_theorem_4, 3);
    for i ∈ eachindex(β_theorem_4)
        β_value = β_theorem_4[i]
    
        # create a copy of the test_note instance
        model = deepcopy(test_note); # Why? check out: https://docs.julialang.org/en/v1/base/copy/#Base.deepcopy
        
        ### BEGIN SOLUTION
        model.rate = β_value*test_note.rate
        model.T = 20.0;
        ### END SOLUTION
        
        # compute: use short-cut syntax and compute the price
        perturbed_price = model |> discount_model |> x-> x.price
        
        # capture: put data in simulation_results_thm4_array
        simulation_results_thm4_array[i,1] = β_value;
        simulation_results_thm4_array[i,2] = 100*((model.rate - test_note.rate)/(test_note.rate));    # percentage change in yield
        simulation_results_thm4_array[i,3] = 100*((model.price - VB20)/(VB20)); # col 2: percentage change in the price of the note
    end
    pretty_table(simulation_results_thm4_array, column_labels=["β","Δy (%)","ΔPrice (%)"] , table_format = TextTableFormat(borders = text_table_borders__simple))
end

=========== ========== =============
         β     Δy (%)   ΔPrice (%) 
=========== ========== =============
       0.8      -20.0      5.23257
  0.866667   -13.3333       3.4552
  0.933333   -6.66667      1.71123
       1.0        0.0          0.0
   1.06667    6.66667     -1.67914
   1.13333    13.3333     -3.32681
       1.2       20.0     -4.94364
=========== ========== =============


### Do your simulation results support the Theorem 4?
1. From the table above, do the Theorem 4 simulations support Malkiel's hypothesis, i.e., that the price change is asymmetric?
2. Holding the coupon rate and the starting yield fixed, would you expect the price asymmetry to increase or decrease with the maturity of the note or bond?

___

## Task 3: Simulate Theorem 5 of Malkiel for a Treasury note
In this task, let's look at the role of the coupon rate.

> __Idea.__ To simulate the impact of changes in the yield (discount) and coupon rate on price, let's perturb the effective nominal discount rate $y$ for a `high`, `nominal`, and `low` coupon rate, with all other values held constant. We'll generate a new rate of the form $y\leftarrow\beta\cdoty$, where $\beta$ is a perturbation value; if $\beta<1$ the perturbed interest rate is _less than_ the nominal rate, if $\beta=1$ the perturbed interest rate is _equal to_ the nominal rate, and if $\beta>1$ the perturbed interest rate is _greater than_ the nominal rate.

First, specify the number (odd) of perturbation values in the `number_of_samples_theorem_5` variable. Next, specify the lower bound in the `β₁` variable and the upper bound in the `β₂` variable. Finally, compute the perturbation array (stored in the `β_theorem_5::Array{Float64,1}` variable) using the [range function](https://docs.julialang.org/en/v1/base/math/#Base.range) in combination with the [Julia pipe |> operator](https://docs.julialang.org/en/v1/manual/functions/#Function-composition-and-piping), and the [collect function](https://docs.julialang.org/en/v1/base/collections/#Base.collect-Tuple{Type,%20Any}).

In [9]:
number_of_samples_theorem_5 = 3;
β₁ = 0.8;
β₂ = 1.2;
β_theorem_5 = range(β₁, stop = β₂, length = number_of_samples_theorem_5) |> collect;

Your job is to complete the implementation of the simulation of Theorem 5 started below and analyze the simulation results.

In [10]:
simulation_results_thm5_array = let

    simulation_results_thm5_array = Array{Float64,2}(undef, number_of_samples_theorem_5, number_of_samples_theorem_5);
    for i ∈ eachindex(β_theorem_5)
        
        β_outer = β_theorem_5[i]
        
        # create a copy of the test_note instance
        model = deepcopy(test_note);
        
        ### BEGIN SOLUTION
        model.coupon = β_outer*test_note.coupon;
        ### END SOLUTION
        
        for j ∈ eachindex(β_theorem_5)
            
            β_inner = β_theorem_5[j];
            
            ### BEGIN SOLUTION
            model.rate = β_inner*test_note.rate;
            ### END SOLUTION

            # compute: use short-cut syntax and compute the price
            perturbed_price = model |> discount_model |> x-> x.price
            
            # compute: the percentage difference between the nominal and perturbed price
            simulation_results_thm5_array[i,j] = ((perturbed_price - nominal_computed_price)/nominal_computed_price)*100
        end
    end
    simulation_results_thm5_array; # return
end;

`Unhide` the code block below to see how we visualized the `simulation_results_thm5_array` data array using [the `pretty_table(...)` function exported from the PrettyTables.jl package](https://github.com/ronisbr/PrettyTables.jl). 

> __Caution__: This code assumes `number_of_samples_theorem_5 = 3`; if you have modified the `number_of_samples_theorem_5` variable, you will need to update the logic that generates the table.

So what do we see?

In [11]:
let
    # build a pretty table to display the results -
    (R,C) = size(simulation_results_thm5_array)
    pretty_table_data = Array{Any,2}(undef, R, C+1)
    
    # first col holds labels -
    for i ∈ 1:R
        if (i == 1)
            pretty_table_data[i,1] = "-20% coupon";
        elseif (i == 3)
            pretty_table_data[i,1] = "+20% coupon";
        else
            pretty_table_data[i,1] = "nominal coupon";
        end
    end
    
    for i = 1:R
        for j = 1:C
            pretty_table_data[i,j+1] = simulation_results_thm5_array[i,j]
        end
    end
    
    header_data = (["", "-20% yield", "nominal yield", "+20% yield"])
    pretty_table(pretty_table_data, column_labels=header_data, table_format = TextTableFormat(borders = text_table_borders__simple))
end

================= ============ =============== =============
                   -20% yield   nominal yield   +20% yield 
================= ============ =============== =============
     -20% coupon     0.109757        -1.83398     -3.73638
  nominal coupon      1.96352             0.0     -1.92189
     +20% coupon      3.81729         1.83398    -0.107394
================= ============ =============== =============


### Do your simulation results support the Theorem 5?
1. From the table above, do the Theorem 5 simulations support Malkiel's hypothesis, i.e., that high coupon instruments are less sensitive to changes in yield (discount rate)? __Hint:__ compare the center square with the corners, what do the differences in value suggest?
___

## Summary

This example used a controlled coupon-security pricing model to examine the asymmetric and coupon-dependent interest-rate sensitivity described by Malkiel's fourth and fifth theorems.

> __Key Takeaways__
>
> * __Coupon-security prices are discounted cash-flow values:__ The note price combines the present values of every coupon and the principal repayment under one stated yield convention.
> * __The price-yield relationship is convex:__ For equal-sized yield changes, a yield decrease raises price by more than an equal yield increase lowers it.
> * __Higher coupons reduce percentage sensitivity when other inputs are fixed:__ Earlier and larger coupon cash flows shorten the effective timing of value and reduce the price response to a given yield change.

These simulations demonstrate implications of the assumed cash-flow model; they do not, by themselves, constitute an empirical test using independent market observations.

___


## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance.  Only risk capital that is not required for living expenses.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___